# Data Warehouse — Sales Analytics
## Storytelling Project

Storytelling hasil *data warehouse* skema bintang di **Neon PostgreSQL** (schema `dwh`),
di-orkestrasi **Apache Airflow**, dan divisualisasikan di **Metabase**.

| Aspek | Detail |
|---|---|
| Sumber data | (1) CSV harian 2 tanggal · (2) MariaDB streaming `stg_categories` |
| Warehouse | Neon PostgreSQL — schema `stg` + `dwh` |
| Orkestrasi | Apache Airflow (DAG `dag_manuel`, conn `neon_manuel`) |
| Dashboard | Metabase ("Dashboard Penjualan") |
| **Total transaksi (`fact_sales`)** | **99.051** |
| Rentang data | Jan – 9 Mei 2018 |

---
## 1. Arsitektur — Satu Project, Dua Source

```
  (1) CSV (zip, 2 tanggal) --[Airflow]------+
                                            |
                                            v
                              Staging (stg) --> Transform/SCD --> Star Schema (dwh) --> Metabase
                                            ^
  (2) MariaDB stg_categories --[CDC/JDBC]---+
      (schema dna_project)
```

- **Pipeline atas (batch)** — CSV 2 tanggal di-load Airflow ke `stg`, lalu dibentuk dimensi & fact di `dwh`. **Selesai.**
- **Pipeline bawah (streaming)** — tabel `stg_categories` dari MariaDB (`dna_project`) di-stream ke Postgres yang sama via CDC/JDBC Sink.

> Catatan: dokumen mapping menulis schema `dm`; implementasi di Neon memakai `dwh`.

---
## 2. Skema Bintang (ERD)

![ERD Star Schema](docs/erd.png)

| Tabel | Peran | Surrogate key | SCD | Baris |
|---|---|---|---|---:|
| `fact_sales` | Fact | (4 FK + `sales_id`) | – | **99.051** |
| `dim_time` | Dimensi waktu | `sk_date` | Type 0 | 36.890 |
| `dim_product` | Dimensi produk | `sk_product` | Type 1 | 452 |
| `dim_customer` | Dimensi pelanggan | `sk_customer` | Type 2 | 98.759 |
| `dim_employee` | Dimensi pegawai | `sk_employee` | Type 2 | 23 |

Label sumbu Metabase (`Sk Customer → Customer City Name`, `Sk Product → Product Name`,
`Sk Date → Date: Bulan`) membuktikan relasi FK fact → dimensi terbaca benar.

---
## 3. Hasil Pipeline Airflow

DAG `dag_manuel` berhasil dijalankan end-to-end — semua task **success**:
`load_staging` → `dim_product` / `dim_customer` / `dim_employee` → `fact_sales`.

![Airflow DAG Sukses](docs/airflow.jpeg)

| Tanggal | Mode | `fact_sales` | Catatan |
|---|---|---:|---|
| 2018-05-08 | full | 98.255 | 0 surrogate key NULL |
| 2018-05-09 | incremental | 99.051 (+796) | dimensi lain tidak berubah |
| 2018-05-09 | trigger ulang | 99.051 | idempotent — tidak ada duplikat |

---
## 4. Dashboard Metabase

![Dashboard Penjualan](docs/metabase.jpeg)

Dashboard "Dashboard Penjualan" berisi 4 chart. Semua metrik memakai **jumlah transaksi (count)**,
bukan revenue, karena `total_price` di sumber bernilai 0.

**Insight 1 — Total Transaksi: 99.051.** Cocok dengan jumlah baris `fact_sales` di Neon — bukti load end-to-end berhasil.

**Insight 2 — Top 10 Produk Terlaris.** Produk teratas (Thyme - Lemon Fresh, Olives - Kalamata, Salmon - Atlantic, dst) jumlahnya sangat berdekatan (~253–268) — tidak ada produk yang mendominasi.

**Insight 3 — Top 10 Kota.** Semua pelanggan di **1 negara (US), 96 kota**; chart dibuat per kota. Top 10 kota (Fort Wayne, Columbus, Tucson, dst) berdekatan (~1.079–1.142 transaksi) — pasar terdiversifikasi.

**Insight 4 — Tren per Bulan.** Jan 23.893 · Feb 21.420 · Mar 24.189 · Apr 22.635 · **Mei 6.914**. Angka Mei turun **bukan karena penjualan anjlok**, tapi karena data hanya sampai 9 Mei (bulan belum penuh).

---
## 5. Catatan Kualitas Data

1. **`TotalPrice` = 0** di sumber → metrik dashboard memakai **jumlah transaksi (count)**, bukan revenue.
2. **Rentang data terbatas** (Jan–9 Mei 2018) → bulan Mei belum penuh; analisis musiman belum valid.
3. **Format tanggal tidak seragam** antar batch (`BirthDate`, `HireDate`) → dinormalisasi di loader sebelum SCD-2 supaya tidak tercipta versi histori palsu.
4. **Idempotensi terverifikasi** → `fact_sales` hapus-lalu-insert per `sk_date`; trigger ulang tanggal yang sama tidak menambah baris.
5. **Geografi homogen** → semua pelanggan di 1 negara (US, 96 kota); chart geografis hanya bermakna di level kota.

---
## 6. Kesimpulan & Rekomendasi

**Kesimpulan**
- Warehouse berhasil dibangun: **99.051 transaksi**, skema bintang, pipeline ter-orkestrasi Airflow, idempotent.
- Penjualan terdistribusi merata antar produk maupun kota → portofolio sehat, risiko konsentrasi rendah.
- SCD diterapkan benar (Type 0/1/2) sesuai dokumen mapping; relasi star schema terbaca benar di Metabase.

**Rekomendasi**
- Lengkapi rentang data hingga >= 1 tahun penuh agar analisis musiman valid.
- Perbaiki kualitas `TotalPrice` di sumber agar metrik revenue bisa ditampilkan.
- Selesaikan pipeline streaming MariaDB (CDC) agar kategori selalu real-time.

---
*Mini project Data Warehouse — Manuel.*